In [1]:
pip install torch-pruning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 47.8 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.

In [2]:
from torchvision.models import resnet18, ResNet18_Weights
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import time, copy, os
from torch.utils.data import random_split
import torch.optim as optim
import torch_pruning as tp
import gzip, io
import numpy as np

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])

train_full = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform)
test_ds = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform)

train_ds, val_ds = random_split(train_full, [45000, 5000], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, 10)
model = model.to(device)

100%|██████████| 170M/170M [00:16<00:00, 10.2MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 215MB/s]


In [4]:
def accuracy(y_pred, y):
    classes = y_pred.argmax(dim=1)
    return (classes == y).float().mean()

def train(model, loss, opt, n_epochs, lr, train_loader, val_loader):
    loss_history = []
    best_res_accuracy = -1.0
    best_weights = None
    count_accuracy_same = 0
    max_count_accuracy_same = 10
    optimizer = opt(model.parameters(), lr=lr)
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss_res = loss(y_pred, y_batch)
            optimizer.zero_grad()
            loss_res.backward()
            optimizer.step()
            loss_history.append(loss_res.item())
        model.eval()
        with torch.no_grad():
            correct = total = 0
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch).argmax(dim=1)
                correct += (pred == y_batch).sum().item()
                total += y_batch.size(0)
            new_accuracy = correct / total
        if new_accuracy > best_res_accuracy:
            best_res_accuracy = new_accuracy
            best_weights = copy.deepcopy(model.state_dict())
            count_accuracy_same = 0
        else:
            count_accuracy_same += 1
        print(f"epoch {epoch}: val_acc={new_accuracy:.4f}")
        if count_accuracy_same == max_count_accuracy_same:
            print(f"early stopped on epoch: {epoch}. best accuracy: {best_res_accuracy:.4f}")
            break
    else:
        print(f"Total epochs: {n_epochs}. best accuracy: {best_res_accuracy:.4f}")
    model.load_state_dict(best_weights)
    return best_res_accuracy

In [5]:
result = train(
    model=model,
    loss=nn.CrossEntropyLoss(),
    opt=optim.Adam,
    n_epochs=15,
    lr=0.001,
    train_loader=train_loader,
    val_loader=val_loader,
)
torch.save(model.state_dict(), "resnet18_cifar_baseline.pt")

epoch 0: val_acc=0.8232
epoch 1: val_acc=0.8432
epoch 2: val_acc=0.8448
epoch 3: val_acc=0.8672
epoch 4: val_acc=0.8716
epoch 5: val_acc=0.8750
epoch 6: val_acc=0.8698
epoch 7: val_acc=0.8630
epoch 8: val_acc=0.8444
epoch 9: val_acc=0.8790
epoch 10: val_acc=0.8678
epoch 11: val_acc=0.8812
epoch 12: val_acc=0.8822
epoch 13: val_acc=0.8782
epoch 14: val_acc=0.8774
Total epochs: 15. best accuracy: 0.8822


In [6]:
@torch.no_grad()
def latency(model, batch=64, count=200):
    model.eval()
    x = torch.randn(batch, 3, 32, 32, device=device)
    for _ in range(50):
        _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(count):
        _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / count

@torch.no_grad()
def test_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

def compress_metric(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    raw = buf.getbuffer().nbytes
    gz = len(gzip.compress(buf.getvalue(), compresslevel=9))
    return raw / 1e6, gz / 1e6

In [7]:
print('result baseline')
print(f'accuracy: {test_accuracy(model, test_loader)}')
print(f'latency: {latency(model)}')
print(f'compress: {compress_metric(model)}')

result baseline
accuracy: 0.8728
latency: 15.538293915000168
compress: (44.774539, 41.501416)


In [8]:
def load_baseline():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(512, 10)
    m.load_state_dict(torch.load("resnet18_cifar_baseline.pt", map_location=device))
    return m.to(device)

# Magnitude pruning

In [9]:
def magnitude_pruning(model, amount=0.5):
    will_pruned = []
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            will_pruned.append((module, "weight"))
    prune.global_unstructured(
        will_pruned,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    return model

def adaptive_magnitude(
    model,
    sp_t, 
    steps,
    fn_func
):
    step_amount = 1 - (1 - sp_t) ** (1 / steps)
    for s in range(steps):
        magnitude_pruning(model, amount=step_amount)
        print(f"step {s+1}/{steps}")
        if fn_func is not None:
            fn_func(model)
    return model

In [10]:
magnitude_model = load_baseline()

def fn_func_mp(m):
    train(model=m, 
          loss=nn.CrossEntropyLoss(),
          opt=optim.Adam,
          n_epochs=2, 
          lr=0.001,
          train_loader=train_loader,
          val_loader=val_loader,
        )

adaptive_magnitude(
    model=magnitude_model, 
    sp_t=0.7, 
    steps=5,
    fn_func=fn_func_mp
)

train(model=magnitude_model,
      loss=nn.CrossEntropyLoss(),
      opt=optim.Adam,
      n_epochs=15,
      lr=0.001,
      train_loader=train_loader,
      val_loader=val_loader
)

for m in magnitude_model.modules():
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        try:
            prune.remove(m, "weight")
        except:
            pass

torch.save(magnitude_model.state_dict(), "resnet18_magnitude_70.pt")

step 1/5
epoch 0: val_acc=0.8860
epoch 1: val_acc=0.8692
Total epochs: 2. best accuracy: 0.8860
step 2/5
epoch 0: val_acc=0.8858
epoch 1: val_acc=0.8840
Total epochs: 2. best accuracy: 0.8858
step 3/5
epoch 0: val_acc=0.8904
epoch 1: val_acc=0.8790
Total epochs: 2. best accuracy: 0.8904
step 4/5
epoch 0: val_acc=0.8900
epoch 1: val_acc=0.8930
Total epochs: 2. best accuracy: 0.8930
step 5/5
epoch 0: val_acc=0.8948
epoch 1: val_acc=0.9002
Total epochs: 2. best accuracy: 0.9002
epoch 0: val_acc=0.8958
epoch 1: val_acc=0.8976
epoch 2: val_acc=0.8892
epoch 3: val_acc=0.8960
epoch 4: val_acc=0.8912
epoch 5: val_acc=0.8914
epoch 6: val_acc=0.8870
epoch 7: val_acc=0.8942
epoch 8: val_acc=0.8962
epoch 9: val_acc=0.8930
epoch 10: val_acc=0.8954
epoch 11: val_acc=0.8934
early stopped on epoch: 11. best accuracy: 0.8976


In [11]:
print('result adaptive_magnitude')
print(f'accuracy: {test_accuracy(magnitude_model, test_loader)}')
print(f'latency: {latency(magnitude_model)}')
print(f'compress: {compress_metric(magnitude_model)}')

result adaptive_magnitude
accuracy: 0.8897
latency: 15.520327125000222
compress: (44.774539, 16.75688)


# Structured channel pruning

In [12]:
structured_model = load_baseline()

def structured_channel_prune(model, sp_t=0.3, steps=1):
    model.eval()
    example_inputs = torch.randn(1, 3, 32, 32).to(device)

    importance = tp.importance.MagnitudeImportance(p=1)

    pruner = tp.pruner.MagnitudePruner(
        model,
        example_inputs,
        importance=importance,
        pruning_ratio=sp_t,
        ignored_layers=[model.fc],
        # iterative_steps=steps,
        global_pruning=False,
    )

    pruner.step()

    return model

params_before = sum(p.numel() for p in structured_model.parameters())
print(f"params before pruning {params_before:,}")

structured_channel_prune(structured_model, sp_t=0.3)

params_after = sum(p.numel() for p in structured_model.parameters())
print(f"params after pruning:  {params_after:,}. Consist of ({100*params_after/params_before}%) from baseline")

acc_before = test_accuracy(structured_model, val_loader)
print(f"test_accuracy after prune (before fine-tune): {acc_before}")

train(model=structured_model,
      loss=nn.CrossEntropyLoss(),
      opt=torch.optim.Adam,
      n_epochs=15,
      lr=0.001,
      train_loader=train_loader,
      val_loader=val_loader)

print(f"final test_acc: {test_accuracy(structured_model, test_loader)}")

torch.save(structured_model.state_dict(), "resnet18_structured_50.pt")

params before pruning 11,173,962
params after pruning:  5,459,866. Consist of (48.86239992582756%) from baseline
test_accuracy after prune (before fine-tune): 0.5614
epoch 0: val_acc=0.8708
epoch 1: val_acc=0.8654
epoch 2: val_acc=0.8660
epoch 3: val_acc=0.8814
epoch 4: val_acc=0.8820
epoch 5: val_acc=0.8758
epoch 6: val_acc=0.8806
epoch 7: val_acc=0.8772
epoch 8: val_acc=0.8660
epoch 9: val_acc=0.8766
epoch 10: val_acc=0.8760
epoch 11: val_acc=0.8720
epoch 12: val_acc=0.8798
epoch 13: val_acc=0.8764
epoch 14: val_acc=0.8600
early stopped on epoch: 14. best accuracy: 0.8820
final test_acc: 0.8733


In [13]:
print('result structured_channel_prune')
print(f'accuracy: {test_accuracy(structured_model, test_loader)}')
print(f'latency: {latency(structured_model)}')
print(f'compress: {compress_metric(structured_model)}')

result structured_channel_prune
accuracy: 0.8733
latency: 10.787903684999947
compress: (21.904139, 20.279641)
